# MedAI vLLM - FIXED

**Runtime -> GPU (T4)**

Upload to Colab and run all cells.

In [ ]:
# Install dependencies
!pip install vllm transformers fastapi pyngrok uvicorn pydantic -q 2>/dev/null
print("Installed!")

In [ ]:
# Kill old processes
import subprocess, os
subprocess.run("pkill -f ngrok || true", shell=True)
os.system("rm -rf ~/.ngrok2/tunnels.yml 2>/dev/null || true")
print("Cleaned!")

In [ ]:
# Setup ngrok
from pyngrok import ngrok
ngrok.set_auth_token("3BM0sCxW0eDMEgAhshblG0Q6cP8_3rdRpdmFj8vkh8oojfZvA")
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
print("Ngrok ready!")

In [ ]:
# Load vLLM
import sys
for m in list(sys.modules.keys()):
    if 'vllm' in m or 'transformers' in m:
        del sys.modules[m]

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print("Loading vLLM...")

llm = LLM(
    model=MODEL,
    tensor_parallel_size=1,
    max_model_len=2048,
    gpu_memory_utilization=0.5,
    dtype="float16"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print("vLLM Ready!")

In [ ]:
# Create FastAPI app
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import uvicorn, threading

app = FastAPI()

class Message(BaseModel):
    role: str
    content: str

class Request(BaseModel):
    model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    messages: List[Message]
    temperature: float = 0.7
    max_tokens: int = 300

@app.post("/v1/chat/completions")
async def chat(req: Request):
    try:
        msgs = [{"role": m.role, "content": m.content} for m in req.messages]
        
        # Convert to prompt (FIXED: use tokenize=False)
        prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        
        # Generate
        outputs = llm.generate([prompt], SamplingParams(
            temperature=req.temperature,
            max_tokens=req.max_tokens,
            stop=["<|im_end|>"]
        ))
        response_text = outputs[0].outputs[0].text
        
        return {
            "id": "chatcmpl-fix",
            "object": "chat.completion",
            "created": 1234567890,
            "model": req.model,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": response_text},
                "finish_reason": "stop"
            }]
        }
    except Exception as e:
        return {"error": str(e)}

@app.get("/health")
def health(): return {"status": "ok", "service": "vLLM"}

@app.get("/")
def root(): return {"message": "MedAI vLLM Fixed"}

print("API Ready!")

In [ ]:
# Start tunnel & server
import time
url = ngrok.connect(8000, bind_tls=True)
print(f"URL: {url}")
print()
print("="*60)
print(f"VLLM_API_URL={url}/v1/chat/completions")
print("="*60)

def run(): uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")
threading.Thread(target=run, daemon=True).start()
time.sleep(3)
print("Server running!")